# Health Eat — 학습 → 예측 → Kaggle 제출

**실행 순서**: 위에서 아래로 셀을 순서대로 실행하세요.  
**런타임**: GPU (T4 이상) 권장 — 런타임 → 런타임 유형 변경 → T4 GPU

| 단계 | 내용 |
|---|---|
| 1 | 환경 설정 (repo clone + 패키지 설치) |
| 2 | Kaggle 인증 (Colab Secrets) |
| 3 | 데이터 다운로드 |
| 4 | 학습 (`train.py`) |
| 5 | 예측 + 제출 파일 생성 (`predict.py`) |
| 6 | Kaggle 제출 |

In [ ]:
# ═══════════════════════════════════════════════════
# 실험 설정 — 매 실험마다 여기만 수정
# ═══════════════════════════════════════════════════
from pathlib import Path

BRANCH          = 'feature/synth-aug-400ep'    # 실험 브랜치
SUBMIT_VERSION  = 'v5-synth-aug'               # 버전 태그 — 파일명·WandB 런 이름에 사용
SUBMIT_MESSAGE  = 'v5 synth aug 640px 400ep'   # Kaggle 제출 메시지

BUILD_SYNTH     = False   # True: 합성 데이터 생성  /  False: 스킵
LOAD_FROM_WANDB = False   # True: WandB에서 모델 불러오기  /  False: 처음부터 학습

# ── 아래는 자동 설정 (수정 불필요) ─────────────────
LOCAL_MODEL         = Path(f'outputs/checkpoints/best-{SUBMIT_VERSION}.pt')
WANDB_ARTIFACT_NAME = f'best-{SUBMIT_VERSION}:latest'
print(f'버전: {SUBMIT_VERSION}  |  브랜치: {BRANCH}  |  모델: {LOCAL_MODEL.name}')


## 1. 환경 설정

In [ ]:
import os

REPO_URL    = 'https://github.com/beomjinkim2000/Code_IT_Team_1_FirstProject.git'
PROJECT_DIR = '/content/Code_IT_Team_1_FirstProject'
# BRANCH는 위 설정 셀에서 정의

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} checkout {BRANCH}
    !git -C {PROJECT_DIR} pull origin {BRANCH}

%cd {PROJECT_DIR}


In [ ]:
!pip install -q \
    ultralytics \
    torchmetrics[detection] \
    albumentations \
    kaggle \
    pyyaml \
    tqdm \
    iterative-stratification \
    ensemble-boxes

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Kaggle 인증

**최초 1회 설정** (이후 재실행 시 자동 적용)

1. 왼쪽 사이드바 **🔑 아이콘** 클릭
2. **`KAGGLE_USERNAME`** → Kaggle 닉네임 입력 후 저장
3. **`KAGGLE_KEY`** → [Kaggle 계정](https://www.kaggle.com/settings/account) → API → **Create New Token** → kaggle.json 열어서 `key` 값 복사 후 저장
4. 각 Secret의 **노트북 액세스 허용** 토글 ON

In [ ]:
import json
import os
from google.colab import userdata

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key      = userdata.get("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"]      = kaggle_key

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

print(f"Kaggle 인증 완료 — 사용자: {kaggle_username}")

## 2-1. WandB 인증

**최초 1회 설정** (이후 재실행 시 자동 적용)

1. 왼쪽 사이드바 **🔑 아이콘** 클릭
2. **`WANDB_API_KEY`** → [wandb.ai/settings](https://wandb.ai/settings) → API keys → 키 복사 후 저장
3. **노트북 액세스 허용** 토글 ON

> 설정하지 않으면 WandB 로그 없이 학습만 진행됩니다. (선택사항)

In [ ]:
import os
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    if wandb_key:
        os.environ['WANDB_API_KEY']  = wandb_key
        os.environ['WANDB_ENTITY']   = 'health-eat-pill-detection'
        os.environ['WANDB_PROJECT']  = 'health-eat-pill-detection'
        print('WandB 인증 완료')
    else:
        print('WANDB_API_KEY 미설정 — WandB 로그 비활성화')
except Exception:
    print('WANDB_API_KEY 미설정 — WandB 로그 비활성화')


## 3. 데이터 다운로드

In [ ]:
from pathlib import Path
import zipfile

COMPETITION_ID = "ai11-level1-project"

raw_data_path = Path("data/raw/ai11-level1-project")
dataset_path  = raw_data_path / "sprint_ai_project1_data"
required_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_dirs):
    print("데이터 이미 존재 — 다운로드 건너뜀")
else:
    raw_data_path.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c {COMPETITION_ID} -p {raw_data_path}
    for zf in raw_data_path.glob("*.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(raw_data_path)
        zf.unlink()
    print("압축 해제 완료")

print("\n데이터 구조:")
for d in required_dirs:
    count = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {d.name}: {count}개")

## 3-1. 합성 데이터 생성 (최초 1회)

- `BUILD_SYNTH = True` → GrabCut 세그멘테이션 후 합성 이미지 생성 + WandB artifact 업로드
- `BUILD_SYNTH = False` → 이미 생성된 경우 스킵 (학습 시 WandB artifact에서 자동 다운로드)
- 학습에 합성 데이터를 쓰려면 아래 셀 12의 `train.py` 명령에 `--synth_data data/augmented/synth` 추가

In [ ]:
# BUILD_SYNTH 는 위 설정 셀(cell 1)에서 정의
if BUILD_SYNTH:
    import yaml as _yaml
    with open('configs/default.yaml') as _f:
        _img_size = _yaml.safe_load(_f)['train']['img_size']
    print(f'synth 해상도: {_img_size}px')
    !python scripts/build_synth_dataset.py \
        --data_root data/raw/ai11-level1-project/sprint_ai_project1_data \
        --out_dir   data/augmented \
        --n_synth   3000 \
        --pill_counts 3 4 \
        --img_size {_img_size}
else:
    print('BUILD_SYNTH = False — 합성 데이터 생성 스킵')


## 3-1-1. pill_bank 샘플 확인

GrabCut으로 오려낸 알약 크롭 20개를 확인합니다.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

SAMPLE_DIR = Path('data/augmented/synth/pill_bank_samples')

if not SAMPLE_DIR.exists():
    print('샘플 없음 — 3-1 셀에서 BUILD_SYNTH=True 로 먼저 생성하세요')
else:
    samples = sorted(SAMPLE_DIR.glob('*.png'))
    cols = 5
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.reshape(-1)
    for ax, path in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        cat_id = path.stem.split('_cat')[-1]
        ax.imshow(img)
        ax.set_title(f'cat {cat_id}', fontsize=9)
        ax.axis('off')
    for ax in axes[len(samples):]:
        ax.axis('off')
    plt.suptitle(f'pill_bank 샘플 ({len(samples)}개)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3-2. 합성 데이터 시각화

- 알약 수별 이미지 분포 확인
- 샘플 이미지 + 바운딩 박스 시각화

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

SYNTH_ROOT = Path('data/augmented/synth')
ANN_PATH   = SYNTH_ROOT / 'annotations.json'
IMG_DIR    = SYNTH_ROOT / 'images'

if not ANN_PATH.exists():
    print('합성 데이터 없음 — 3-1 셀에서 BUILD_SYNTH=True 로 먼저 생성하세요')
else:
    with open(ANN_PATH, encoding='utf-8') as f:
        coco = json.load(f)

    # 이미지별 알약 수 집계
    pill_count: Counter = Counter()
    for ann in coco['annotations']:
        pill_count[ann['image_id']] += 1
    count_dist: Counter = Counter(pill_count.values())
    total = len(coco['images'])

    print(f'합성 이미지 총 {total}장 | 어노테이션 {len(coco["annotations"])}개')
    print(f'평균 알약 수: {len(coco["annotations"]) / max(total, 1):.2f}개/장')
    print()
    for n in sorted(count_dist):
        cnt = count_dist[n]
        print(f'  {n}알약: {cnt:4d}장 ({cnt/total*100:.1f}%)')

    # ── 그래프 1: 알약 수 분포 막대 ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    keys   = sorted(count_dist.keys())
    values = [count_dist[k] for k in keys]
    axes[0].bar([str(k) for k in keys], values, color='steelblue', edgecolor='black')
    for x, v in zip(range(len(keys)), values):
        axes[0].text(x, v + 5, f'{v}\n({v/total*100:.0f}%)', ha='center', va='bottom', fontsize=10)
    axes[0].set_title('알약 수 분포 (실제 배치 결과)')
    axes[0].set_xlabel('이미지당 알약 수')
    axes[0].set_ylabel('이미지 수')
    axes[0].grid(True, alpha=0.3, axis='y')

    # 클래스별 분포
    cat_counter: Counter = Counter()
    for ann in coco['annotations']:
        cat_counter[ann['category_id']] += 1
    cats   = sorted(cat_counter.keys())
    counts = [cat_counter[c] for c in cats]
    axes[1].bar([str(c) for c in cats], counts, color='coral', edgecolor='black')
    axes[1].set_title(f'클래스별 어노테이션 수 ({len(cats)}종)')
    axes[1].set_xlabel('category_id')
    axes[1].set_ylabel('어노테이션 수')
    axes[1].grid(True, alpha=0.3, axis='y')
    if len(cats) > 20:
        axes[1].set_xticks([])
        axes[1].set_xlabel(f'category_id (총 {len(cats)}종, 레이블 생략)')

    plt.suptitle('합성 데이터 분포', fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── 그래프 2: 샘플 이미지 8장 (알약 수별 2장씩) ──────────────────
    images_by_id = {img['id']: img['file_name'] for img in coco['images']}
    anns_by_img: dict = {}
    for ann in coco['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)

    # 각 알약 수별 샘플 2장 수집
    samples = []
    for n_pills in sorted(count_dist.keys()):
        ids_with_n = [img_id for img_id, cnt in pill_count.items() if cnt == n_pills]
        chosen = random.sample(ids_with_n, min(2, len(ids_with_n)))
        for img_id in chosen:
            samples.append((img_id, images_by_id[img_id], anns_by_img.get(img_id, [])))

    n_samples = len(samples)
    cols = 4
    rows = (n_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1) if n_samples > 1 else [axes]

    COLORS = ['red', 'lime', 'cyan', 'yellow', 'magenta']
    for ax, (img_id, fname, ann_list) in zip(axes, samples):
        img_path = IMG_DIR / fname
        if not img_path.exists():
            ax.axis('off')
            continue
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        for i, ann in enumerate(ann_list):
            x, y, w, h = ann['bbox']
            color = COLORS[i % len(COLORS)]
            rect = patches.Rectangle((x, y), w, h,
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y - 3, f'c{ann["category_id"]}', color=color, fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))
        ax.set_title(f'{len(ann_list)}알약 | {fname}', fontsize=9)
        ax.axis('off')

    for ax in axes[n_samples:]:
        ax.axis('off')

    plt.suptitle('합성 이미지 샘플 (알약 수별 2장)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. 학습

`configs/default.yaml`에서 설정 조정:
- `train.epochs`: 에폭 수 (기본 50)
- `train.batch_size`: 배치 크기 (기본 16, GPU 메모리 부족 시 8로 낮추기)
- `train.lr`: 학습률 (기본 0.001)

In [ ]:
!cat configs/default.yaml

## 4-1. WandB에서 모델 불러오기 (선택)

`LOAD_FROM_WANDB = False` → 처음부터 학습 (기본값)  
`LOAD_FROM_WANDB = True` → `WANDB_ARTIFACT_NAME`에 지정한 artifact 다운로드 후 학습 스킵  

artifact 이름 형식: `best-{버전명}:latest` (예: `best-v5-synth-aug:latest`)

In [ ]:
import glob, os, shutil
import wandb

LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)
SKIP_TRAIN = False

if LOAD_FROM_WANDB:
    entity   = os.environ.get('WANDB_ENTITY',  'health-eat-pill-detection')
    project  = os.environ.get('WANDB_PROJECT', 'health-eat-pill-detection')
    api      = wandb.Api()
    artifact = api.artifact(f'{entity}/{project}/{WANDB_ARTIFACT_NAME}')
    artifact.download('outputs/checkpoints/')
    pts = [p for p in glob.glob('outputs/checkpoints/*.pt') if 'last' not in p and str(LOCAL_MODEL) not in p]
    if pts and not LOCAL_MODEL.exists():
        shutil.copy(pts[0], LOCAL_MODEL)
    print(f'WandB 다운로드 완료 → {LOCAL_MODEL.name}')
    SKIP_TRAIN = True
else:
    print(f'처음부터 학습 → {LOCAL_MODEL.name}')


In [ ]:
from pathlib import Path as _P

# 합성 데이터가 있으면 자동으로 --synth_data 추가
_synth_flag = '--synth_data data/augmented/synth' if _P('data/augmented/synth/annotations.json').exists() else ''

if not SKIP_TRAIN:
    !python train.py \
        --config configs/default.yaml \
        --run-name {SUBMIT_VERSION} \
        --version {SUBMIT_VERSION} \
        {_synth_flag}


In [ ]:
if not SKIP_TRAIN:
    if LOCAL_MODEL.exists():
        print(f'✅ {LOCAL_MODEL.name} 저장 완료 (WandB artifact에도 자동 업로드됨)')
    else:
        print(f'{LOCAL_MODEL.name} 없음 — 학습이 완료됐는지 확인하세요')


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

log_path = Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv')
if not log_path.exists():
    print(f'metrics_{SUBMIT_VERSION}.csv 없음 — 학습을 먼저 실행하세요')
else:
    df = pd.read_csv(log_path)
    has_val_loss = 'val_box_loss' in df.columns

    nrows = 3 if has_val_loss else 2
    fig, axes = plt.subplots(nrows, 3, figsize=(18, 5 * nrows))

    axes[0,0].plot(df['epoch'], df['box_loss'], label='box')
    axes[0,0].plot(df['epoch'], df['cls_loss'], label='cls')
    axes[0,0].plot(df['epoch'], df['dfl_loss'], label='dfl')
    axes[0,0].set_title('Train Loss'); axes[0,0].set_xlabel('Epoch')
    axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(df['epoch'], df['val_mAP_ema'], label='EMA', linewidth=2)
    axes[0,1].plot(df['epoch'], df['val_mAP_raw'], label='raw', linestyle='--', alpha=0.7)
    axes[0,1].set_title('mAP@0.5:0.95 (EMA vs raw)'); axes[0,1].set_xlabel('Epoch')
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    axes[0,2].plot(df['epoch'], df['val_mAP_50_ema'], label='EMA', linewidth=2)
    axes[0,2].plot(df['epoch'], df['val_mAP_50_raw'], label='raw', linestyle='--', alpha=0.7)
    axes[0,2].set_title('mAP@0.5 (EMA vs raw)'); axes[0,2].set_xlabel('Epoch')
    axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

    axes[1,0].plot(df['epoch'], df['lr'], color='green')
    axes[1,0].set_title('Learning Rate'); axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_yscale('log'); axes[1,0].grid(True, alpha=0.3)

    gain = df['val_mAP_ema'] - df['val_mAP_raw']
    axes[1,1].plot(df['epoch'], gain, color='purple')
    axes[1,1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1,1].set_title('EMA 이득 (mAP@0.5:0.95)'); axes[1,1].set_xlabel('Epoch')
    axes[1,1].grid(True, alpha=0.3)

    gain50 = df['val_mAP_50_ema'] - df['val_mAP_50_raw']
    axes[1,2].plot(df['epoch'], gain50, color='orange')
    axes[1,2].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1,2].set_title('EMA 이득 (mAP@0.5)'); axes[1,2].set_xlabel('Epoch')
    axes[1,2].grid(True, alpha=0.3)

    if has_val_loss:
        for col_idx, (key, label) in enumerate([('box', 'Box'), ('cls', 'Cls'), ('dfl', 'DFL')]):
            ax = axes[2, col_idx]
            ax.plot(df['epoch'], df[f'{key}_loss'], label='train', linewidth=2)
            ax.plot(df['epoch'], df[f'val_{key}_loss'], label='val', linestyle='--', linewidth=2)
            ax.set_title(f'{label} Loss: Train vs Val'); ax.set_xlabel('Epoch')
            ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle(f'Training Metrics — {SUBMIT_VERSION}', fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

    best_ema = df.loc[df['val_mAP_ema'].idxmax()]
    best_raw = df.loc[df['val_mAP_raw'].idxmax()]
    gain_val = best_ema['val_mAP_ema'] - best_raw['val_mAP_raw']
    gain_pct = gain_val / best_raw['val_mAP_raw'] * 100 if best_raw['val_mAP_raw'] > 0 else 0
    print(f"best(EMA) epoch={int(best_ema['epoch'])}  mAP@50:95={best_ema['val_mAP_ema']:.4f}  mAP@50={best_ema['val_mAP_50_ema']:.4f}")
    print(f"best(raw) epoch={int(best_raw['epoch'])}  mAP@50:95={best_raw['val_mAP_raw']:.4f}  mAP@50={best_raw['val_mAP_50_raw']:.4f}")
    print(f"EMA 이득: {gain_val:+.4f}  ({gain_pct:+.1f}%)")


## 4-3. 클래스별 F1 추적

best epoch 기준 클래스별 F1을 확인합니다.

| 판단 기준 | 의미 |
|---|---|
| F1 < 0.7 (빨간색) | 해당 클래스가 bottleneck |
| Precision < Recall | 다른 클래스로 오탐(FP)이 많음 |
| Recall < Precision | 알약을 못 잡음(FN) — 데이터 부족 or 해상도 문제 |

→ Q4(YOLO11x), Q6(해상도 1024), Q7(AI Hub 데이터) 실험에서 어떤 클래스가 개선됐는지 비교하세요.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

f1_path      = Path(f'outputs/logs/f1_{SUBMIT_VERSION}.csv')
metrics_path = Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv')

if not f1_path.exists():
    print(f'f1_{SUBMIT_VERSION}.csv 없음 — 학습을 먼저 실행하세요')
else:
    df_f1 = pd.read_csv(f1_path)

    # best EMA epoch 기준
    if metrics_path.exists():
        df_m = pd.read_csv(metrics_path)
        best_epoch = int(df_m.loc[df_m['val_mAP_ema'].idxmax(), 'epoch'])
    else:
        best_epoch = int(df_f1['epoch'].max())

    df_best = df_f1[df_f1['epoch'] == best_epoch].copy()

    if df_best.empty:
        print(f'epoch={best_epoch} 데이터 없음')
    else:
        LOW_F1 = 0.7
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))

        # 클래스별 F1 바 차트 — F1 < 0.7 은 빨간색
        colors = ['red' if f < LOW_F1 else 'steelblue' for f in df_best['f1']]
        axes[0].barh(df_best['class_id'].astype(str), df_best['f1'], color=colors)
        axes[0].axvline(LOW_F1, color='red', linestyle='--', label=f'F1={LOW_F1} 기준선')
        axes[0].set_title(f'클래스별 F1 (epoch={best_epoch}, best EMA)')
        axes[0].set_xlabel('F1'); axes[0].set_ylabel('class_id')
        axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='x')

        # F1 낮은 클래스의 에폭 추이 (최대 10개)
        low_classes = df_best[df_best['f1'] < LOW_F1]['class_id'].tolist()
        if low_classes:
            for cls_id in low_classes[:10]:
                sub = df_f1[df_f1['class_id'] == cls_id]
                axes[1].plot(sub['epoch'], sub['f1'], label=f'cls {cls_id}')
            axes[1].axhline(LOW_F1, color='red', linestyle='--', alpha=0.5)
            axes[1].set_title(f'F1 < {LOW_F1} 클래스 에폭 추이')
            axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
            axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
        else:
            axes[1].text(0.5, 0.5, f'모든 클래스 F1 ≥ {LOW_F1}', ha='center', va='center',
                         transform=axes[1].transAxes, fontsize=14, color='green')
            axes[1].set_title('F1 추이')

        plt.suptitle(f'클래스별 F1 — {SUBMIT_VERSION}', fontsize=13)
        plt.tight_layout(); plt.show()

        print(f"=== best epoch={best_epoch} 클래스별 F1 ===")
        print(f"{'cls':>5}  {'precision':>9}  {'recall':>7}  {'f1':>6}  {'판정'}")
        print("-" * 46)
        for _, row in df_best.sort_values('f1').iterrows():
            flag = "⚠️  낮음" if row['f1'] < LOW_F1 else ""
            print(f"{int(row['class_id']):>5}  {row['precision']:>9.4f}  {row['recall']:>7.4f}  {row['f1']:>6.4f}  {flag}")

        n_low = (df_best['f1'] < LOW_F1).sum()
        print(f"\nF1 < {LOW_F1} 클래스: {n_low} / {len(df_best)}개")
        if n_low > 0:
            worst = df_best.loc[df_best['f1'].idxmin()]
            print(f"최저 F1: cls {int(worst['class_id'])}  "
                  f"F1={worst['f1']:.4f}  P={worst['precision']:.4f}  R={worst['recall']:.4f}")
            if worst['precision'] < worst['recall']:
                print("  → Precision < Recall: 다른 클래스로 오탐(FP)이 많음")
            else:
                print("  → Recall < Precision: 알약을 못 잡음(FN) — 데이터 부족 or 해상도 문제")

In [ ]:
import shutil
from pathlib import Path

sub_dir = Path('outputs/submissions')
sub_dir.mkdir(parents=True, exist_ok=True)

if not LOCAL_MODEL.exists():
    print(f'⚠️  {LOCAL_MODEL.name} 없음 — 학습 완료 후 실행하거나 LOAD_FROM_WANDB=True 로 불러오세요')
else:
    # EMA 예측
    !python predict.py --config configs/default.yaml --checkpoint {LOCAL_MODEL} --use-ema
    src = sub_dir / 'submission_v1.csv'
    if src.exists():
        dst = sub_dir / f'submission_{SUBMIT_VERSION}_ema.csv'
        shutil.copy(src, dst)
        print(f'✅ EMA 예측 완료 → {dst.name}')

    # raw 예측
    !python predict.py --config configs/default.yaml --checkpoint {LOCAL_MODEL}
    src = sub_dir / 'submission_v1.csv'
    if src.exists():
        dst = sub_dir / f'submission_{SUBMIT_VERSION}_raw.csv'
        shutil.copy(src, dst)
        print(f'✅ raw 예측 완료 → {dst.name}')


In [ ]:
import pandas as pd
from pathlib import Path

sub_dir = Path('outputs/submissions')
for label, fname in [('EMA', f'submission_{SUBMIT_VERSION}_ema.csv'),
                     ('raw', f'submission_{SUBMIT_VERSION}_raw.csv')]:
    p = sub_dir / fname
    if p.exists():
        df = pd.read_csv(p)
        det = df.groupby('image_id').size()
        print(f"[{label:>3}]  행 수: {len(df):5d}  이미지: {df['image_id'].nunique():4d}"
              f"  탐지/이미지: {det.mean():.2f}  avg score: {df['score'].mean():.4f}"
              f"  max: {df['score'].max():.4f}  min: {df['score'].min():.4f}")
    else:
        print(f'[{label:>3}]  파일 없음: {p}')


## 5-1. 예측 진단

score 분포와 이미지당 탐지 수를 확인합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# EMA 파일 우선, 없으면 raw로 fallback
csv_path = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_ema.csv')
if not csv_path.exists():
    csv_path = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_raw.csv')
if not csv_path.exists():
    print(f'submission_{SUBMIT_VERSION}_ema.csv / _raw.csv 없음 — 예측을 먼저 실행하세요')
else:
    df = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df['score'], bins=50, edgecolor='black', color='steelblue')
    axes[0].axvline(0.25, color='red', linestyle='--', label='conf_threshold=0.25')
    axes[0].set_title('Score Distribution'); axes[0].set_xlabel('score'); axes[0].set_ylabel('Count')
    axes[0].legend()

    det_per_img = df.groupby('image_id').size()
    axes[1].hist(det_per_img, bins=range(0, det_per_img.max() + 2), edgecolor='black', color='steelblue')
    axes[1].set_title('Detections per Image'); axes[1].set_xlabel('Detections'); axes[1].set_ylabel('Images')

    plt.tight_layout(); plt.show()
    print(f"파일: {csv_path.name}")
    print(f"score  평균: {df['score'].mean():.3f}  최솟값: {df['score'].min():.3f}  최댓값: {df['score'].max():.3f}")
    print(f"이미지당 평균 탐지: {det_per_img.mean():.1f}  최대: {det_per_img.max()}  검출된 이미지 수: {len(det_per_img)}")

In [ ]:
# 학습 완료 후 실행 — 모델 없으면 자동 스킵
if not LOCAL_MODEL.exists():
    print(f'⚠️  {LOCAL_MODEL.name} 없음 — 학습 완료 후 실행하거나 LOAD_FROM_WANDB=True 로 불러오세요')
else:
    import sys; sys.path.insert(0, '.')
    import torch, numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from PIL import Image
    from src.utils.config import load_config
    from src.data.dataset import PillDataset, RAW_DATA_ROOT
    from src.data.split import train_val_split
    from src.data.transforms import val_transform
    from src.engine.predict import predict_batch
    from src.engine.postprocess import PostprocessConfig, postprocess_raw_outputs
    from src.models.baseline import build_model

    cfg = load_config()
    img_size = cfg['train']['img_size']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    ckpt = torch.load(LOCAL_MODEL, map_location=device, weights_only=True)
    model = build_model(cfg['data']['nc'])
    model.load_state_dict(ckpt['model_state'])
    model.to(device).eval()

    all_files = sorted((RAW_DATA_ROOT / 'train_images').glob('*.png'))
    _, val_files = train_val_split(all_files, val_ratio=cfg['split']['val_ratio'], seed=cfg['train']['seed'], method='random')
    sample_files = val_files[:4]

    postprocess_cfg = PostprocessConfig(**cfg['postprocess'])
    transform = val_transform(img_size)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, fname in zip(axes, sample_files):
        img = Image.open(RAW_DATA_ROOT / 'train_images' / fname).convert('RGB')
        orig_w, orig_h = img.size
        t_img, _ = transform(img, {'boxes': [], 'labels': [], 'image_id': 0})
        tensor = torch.from_numpy(np.array(t_img)).permute(2,0,1).float().div(255).unsqueeze(0).to(device)
        raw = predict_batch(model, tensor, device)
        preds = postprocess_raw_outputs(raw, image_ids=[0], config=postprocess_cfg)
        ax.imshow(img)
        for box, score, label in zip(preds[0]['boxes'].cpu(), preds[0]['scores'].cpu(), preds[0]['labels'].cpu()):
            x1, y1, x2, y2 = [v * (orig_w if i % 2 == 0 else orig_h) / img_size for i, v in enumerate(box.tolist())]
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none'))
            ax.text(x1, y1-5, f'cls{label.item()} {score:.2f}', color='red', fontsize=7)
        ax.set_title(f'{fname} ({len(preds[0]["boxes"])} boxes)'); ax.axis('off')

    plt.suptitle('Val Predictions (red = predicted box)', fontsize=12)
    plt.tight_layout(); plt.show()


## 5-2. F1 곡선 — 최적 confidence threshold 찾기

현재 `postprocess.conf_threshold=0.25`는 관행적 기본값입니다.
이 셀은 val set 전체를 conf=0.001로 예측한 뒤 conf를 0.05→0.90으로 sweep해서
**F1이 최대인 conf**를 찾습니다.

- `best_conf`가 현재값과 ≥ 0.05 차이나면 → `configs/default.yaml`에 반영 후 재제출
- `skip_box_thr` (WBF)도 같은 방식으로 최적화 가능 ([[concepts/Confidence Score]])

In [ ]:
import sys; sys.path.insert(0, '.')
from src.utils.config import load_config
cfg = load_config()

# 이 셀은 위 '5-1. 예측 진단' 셀 실행 후 실행하세요 (model, val_files, cfg 공유)
from src.engine.evaluate import compute_per_class_f1
from src.engine.postprocess import PostprocessConfig
import numpy as np

nc           = cfg['data']['nc']
current_conf = cfg['postprocess']['conf_threshold']

# val 전체 예측 수집 — conf=0.001 로 낮춰야 PR 곡선 전체 범위 커버
raw_post_cfg = PostprocessConfig(
    conf_threshold=0.001,
    iou_threshold=postprocess_cfg.iou_threshold,
    max_detections=postprocess_cfg.max_detections,
)

from src.data.dataset import PillDataset
dataset = PillDataset(
    image_files=val_files,
    data_root=RAW_DATA_ROOT / 'train_images',
    transform=transform,
    annotation_dir=RAW_DATA_ROOT / 'train_annotations',
)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=8, shuffle=False,
    collate_fn=lambda batch: list(zip(*batch))
)

print("val set 예측 수집 중 (conf=0.001)...")
all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for images, targets in loader:
        imgs = torch.stack([
            torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255)
            for img in images
        ]).to(device)
        raw  = predict_batch(model, imgs, device)
        preds = postprocess_raw_outputs(raw, image_ids=[t['image_id'] for t in targets], config=raw_post_cfg)
        all_preds.extend(preds)
        all_targets.extend(targets)
print(f"수집 완료: {len(all_preds)}개 이미지")

# conf sweep 0.05 → 0.90
conf_range = np.arange(0.05, 0.91, 0.05)
f1_vals, prec_vals, rec_vals = [], [], []
print("F1 곡선 계산 중...")
for conf in conf_range:
    stats = compute_per_class_f1(all_preds, all_targets, num_classes=nc, conf_threshold=float(conf))
    if stats:
        f1_vals.append(np.mean([v['f1']        for v in stats.values()]))
        prec_vals.append(np.mean([v['precision'] for v in stats.values()]))
        rec_vals.append(np.mean([v['recall']    for v in stats.values()]))
    else:
        f1_vals.append(0.0); prec_vals.append(0.0); rec_vals.append(0.0)

best_idx  = int(np.argmax(f1_vals))
best_conf = float(conf_range[best_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(conf_range, f1_vals,   label='F1',        linewidth=2,   color='blue')
axes[0].plot(conf_range, prec_vals, label='Precision',  linestyle='--', color='green')
axes[0].plot(conf_range, rec_vals,  label='Recall',     linestyle='--', color='orange')
axes[0].axvline(best_conf,    color='blue', linestyle=':',  linewidth=1.5, label=f'최적 conf={best_conf:.2f}')
axes[0].axvline(current_conf, color='red',  linestyle='--', linewidth=1.5, label=f'현재 conf={current_conf:.2f}')
axes[0].set_title('F1 / Precision / Recall vs conf threshold')
axes[0].set_xlabel('conf threshold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(rec_vals, prec_vals, 'b-o', markersize=4)
axes[1].set_title('PR 곡선 (conf sweep)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'F1 곡선 — 최적 conf threshold 탐색 ({SUBMIT_VERSION})', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\n현재  conf_threshold : {current_conf:.2f}  (configs/default.yaml postprocess.conf_threshold)")
print(f"최적  conf_threshold : {best_conf:.2f}  (F1={f1_vals[best_idx]:.4f})")
if abs(best_conf - current_conf) >= 0.05:
    print(f"⚠️  default.yaml의 postprocess.conf_threshold를 {best_conf:.2f}로 바꾸고 재제출 권장")
else:
    print(f"✅  현재 conf={current_conf:.2f}이 최적과 가깝습니다 — 변경 불필요")

## 6. Kaggle 제출

In [ ]:
# ── 제출 모델 선택 ─────────────────────────────────────────────────
# 위 비교 확인 후 선택하세요
USE_EMA = True   # True: EMA 모델  /  False: 원본 모델
# ─────────────────────────────────────────────────────────────────

from pathlib import Path
model_label = "EMA" if USE_EMA else "raw"
chosen_csv  = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_{model_label}.csv')

if chosen_csv.exists():
    import pandas as pd
    df = pd.read_csv(chosen_csv)
    print(f'제출 파일: {chosen_csv.name}')
    print(f'행 수: {len(df)}  /  이미지 수: {df["image_id"].nunique()}')
    df.head(5)
else:
    print(f'⚠️  {chosen_csv} 없음 — 예측 셀을 먼저 실행하세요')

In [ ]:
!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f {chosen_csv} \
    -m "{SUBMIT_MESSAGE} [{model_label}]"

print(f'제출 완료! ({model_label} 모델)')

# 제출 후 GPU 세션 자동 해제
from google.colab import runtime
runtime.unassign()

In [ ]:
!kaggle competitions submissions -c {COMPETITION_ID}